# **Extracción Dinámica de Datos Web (Web Scraping) con Python y BeautifulSoup**


En este cuaderno aprenderemos a construir un flujo de Web Scraping para transformar páginas web en conjuntos de datos estructurados y listos para su análisis.

Antes de integrar herramientas de automatización como GitHub Actions o plataformas de visualización como Power BI, el primer paso fundamental ocurre en el entorno de desarrollo: inspeccionar, comprender y extraer la información directamente desde el código fuente del sitio objetivo.

A lo largo de este ejercicio práctico aprenderemos a:

* **Inspeccionar páginas web dinámicas:** Utilizar las herramientas de desarrollador (F12 / DevTools) para analizar la estructura HTML de cualquier sitio y seleccionar los contenedores clave sin depender de plantillas rígidas.

* **Procesar y limpiar datos con Python:** Automatizar la recolección masiva mediante ciclos, limpiar cadenas de texto no deseadas (.text, .strip(), .replace()) y organizar la información en tablas estructuradas utilizando la librería Pandas.

* **Exportar resultados para BI:** Generar archivos CSV estandarizados que servirán como la fuente primaria de datos para la posterior automatización y conexión a tableros de control.



In [1]:

import pandas as pd

# Requests sirve para enviar solicitudes HTTP a servidores web, permite interactuar con páginas web.
import requests

# BeautifulSoup sirve para analizar y extraer datos de documentos HTML y XML.
from bs4 import BeautifulSoup

import pandas as pd

In [2]:
# 1. Hacer la petición a la página objetivo
url = "https://quotes.toscrape.com/tag/life/"

#Envía una petición GET al servidor. El servidor entrega todo el HTML de la página.
response = requests.get(url)

# Garantiza que caracteres especiales se interpreten sin errores.
response.encoding = 'utf-8'

In [3]:
# 2. Convertir el texto HTML a un objeto parseable con BeautifulSoup
# Toma el texto HTML de la respuesta y lo analiza con el motor "html.parser".
# Un objeto parseable es un dato en formato de texto plano que tiene la estructura correcta para ser analizado
texto = BeautifulSoup(response.text, "html.parser")
texto

<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
<title>Quotes to Scrape</title>
<link href="/static/bootstrap.min.css" rel="stylesheet"/>
<link href="/static/main.css" rel="stylesheet"/>
</head>
<body>
<div class="container">
<div class="row header-box">
<div class="col-md-8">
<h1>
<a href="/" style="text-decoration: none">Quotes to Scrape</a>
</h1>
</div>
<div class="col-md-4">
<p>
<a href="/login">Login</a>
</p>
</div>
</div>
<h3>Viewing tag: <a href="/tag/life/page/1/">life</a></h3>
<div class="row">
<div class="col-md-8">
<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
<span class="text" itemprop="text">“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”</span>
<span>by <small class="author" itemprop="author">Albert Einstein</small>
<a href="/author/Albert-Einstein">(about)</a>
</span>
<div class="tags">
            Tags:
            <meta class="keywords" con

In [4]:
# 3. Localizar todas las tarjetas de productos (etiquetas <article class="product_pod">)
# Busca en todo el documento HTML cada etiqueta que cumpla la condición.
# La instrucción afectará únicamente a las etiquetas <article> de la página HTML
# Se filtra para obtener solo los artículos cuya clase sea exactamente product_pod (cada libro tiene una tarjeta de este tipo).

#libros = texto.find_all("article", class_="product_pod")
#libros

# Buscar todas las citas
citas = texto.find_all("div", class_="quote")

In [9]:
datos = []

# Extraer información de cada cita
for cita in citas:

    texto_cita = cita.find("span", class_="text").get_text(strip=True)

    autor = cita.find("small", class_="author").get_text(strip=True)

    etiquetas = [
        tag.get_text(strip=True)
        for tag in cita.find_all("a", class_="tag")
    ]

    datos.append({
        "cita": texto_cita,
        "autor": autor,
        "etiquetas": ", ".join(etiquetas)
    })


In [10]:
# 5. Convertir la lista de diccionarios a un DataFrame (Estructura Tidy Data)
df = pd.DataFrame(datos)
df

,cita,autor,etiquetas
0,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
1,“It is better to be hated for what you are tha...,André Gide,"life, love"
2,“This life is what you make it. No matter what...,Marilyn Monroe,"friends, heartbreak, inspirational, life, love..."
3,"“I may not have gone where I intended to go, b...",Douglas Adams,"life, navigation"
4,"“Good friends, good books, and a sleepy consci...",Mark Twain,"books, contentment, friends, friendship, life"
5,“Life is what happens to us while we are makin...,Allen Saunders,"fate, life, misattributed-john-lennon, plannin..."
6,"“Today you are You, that is truer than true. T...",Dr. Seuss,"comedy, life, yourself"
7,“Life is like riding a bicycle. To keep your b...,Albert Einstein,"life, simile"
8,“Life isn't about finding yourself. Life is ab...,George Bernard Shaw,"inspirational, life, yourself"
9,“Finish each day and be done with it. You have...,Ralph Waldo Emerson,"life, regrets"


In [11]:
df.to_csv("mensajes_vida.csv", index=False)

In [12]:
%%writefile mensajes_vida.py
import requests
from bs4 import BeautifulSoup
import pandas as pd
# Guardar el script completo en un archivo .py local
script_code = """import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://quotes.toscrape.com/tag/life/"

response = requests.get(url)
response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")

citas = soup.find_all("div", class_="quote")

datos = []

for cita in citas:

    texto_cita = cita.find("span", class_="text").get_text(strip=True)

    autor = cita.find("small", class_="author").get_text(strip=True)

    etiquetas = [
        tag.get_text(strip=True)
        for tag in cita.find_all("a", class_="tag")
    ]

    datos.append({
        "cita": texto_cita,
        "autor": autor,
        "etiquetas": ", ".join(etiquetas)
    })

df = pd.DataFrame(datos)

df.to_csv("citas_vida.csv", index=False, encoding="utf-8-sig")

print(f"Se extrajeron {len(df)} citas.")
print("Archivo citas_vida.csv creado correctamente.")
"""

with open("scraper.py", "w", encoding="utf-8") as f:
    f.write(script_code)


Overwriting mensajes_vida.py
